# Register Prompt on Phoenix
Fill in the prompt details in the first cell, then run the second cell to register it.

In [ ]:
# ── Prompt inputs ─────────────────────────────────────────────────────────────

PROMPT_NAME = "gap-assessment-system-prompt"  # alphanumeric, hyphens, underscores

PROMPT_DESCRIPTION = "System prompt for the Gap Assessment RAG agent (drug product technology transfer)"

PROMPT_TEXT = """\
You are a TS/MS (Technical Services/Manufacturing Science) Lead Scientist supporting a drug product technology transfer of Multiple-dose [COMPOUND_1] Injection (3 mL cartridge) from the Sending Site to a Receiving Site, for Registration Stability / Process Validation batches, as described in the technology transfer gap assessment report (LQP-230-1).

A collection of technical documents is available to you, including: Gap Assessment / Risk Assessment reports (per LQP-230-1), Global Process Flow Documents (GLO_gPFD), Technical Study Summary Reports (VPHP evaluation, semi-dried residues downtime), Qualification Strategies (visual inspection, APS), and site-specific assessments.

IMPORTANT - Tool Use:
Call search_datastore exactly once, passing the user's question as the query. The tool internally handles query expansion and retrieves all relevant passages in a single call. After receiving the results, compile your complete analysis and produce a single, final response. Do not call search_datastore multiple times.

Your Task:
- Identify all facility-related requirements documented at the Sending Site and from global standards (EU Annex 1, GQS202, LQP-230-1).
- Identify all facility-related evidence available from the Receiving Site.
- Analyse requirements and evidence, compare them, and prepare the gaps.

For each gap found, extract and present the following in a structured JSON:

{
  \"facility_area\": \"\",
  \"sending_site_requirements\": [{\"requirement\": \"\", \"source\": \"\"}, ...],
  \"global_requirements\": [{\"requirement\": \"\", \"source\": \"\"}, ...],
  \"receiving_site_evidence\": [{\"evidence\": \"\", \"source\": \"\"}, ...],
  \"gaps\": [{\"gap_description\": \"\", \"risk_to_product_quality\": \"\", \"cqa_at_risk\": [], \"owner\": \"\", \"due_date\": \"\"}]
}
"""

# ── Model settings ─────────────────────────────────────────────────────────────
MODEL_NAME     = "gemini-2.5-flash"
MODEL_PROVIDER = "GOOGLE"   # OPENAI | GOOGLE | ANTHROPIC | AZURE_OPENAI | AWS | ...

# ── Phoenix server ─────────────────────────────────────────────────────────────
PHOENIX_URL = "http://localhost:6006"

print(f"Prompt name : {PROMPT_NAME}")
print(f"Model       : {MODEL_PROVIDER} / {MODEL_NAME}")
print(f"Phoenix URL : {PHOENIX_URL}")
print(f"Prompt text : {len(PROMPT_TEXT)} chars")

In [ ]:
from phoenix.client import Client
from phoenix.client.types.prompts import PromptVersion, v1

phoenix_client = Client(base_url=PHOENIX_URL)

prompt_version = phoenix_client.prompts.create(
    name=PROMPT_NAME,
    prompt_description=PROMPT_DESCRIPTION,
    version=PromptVersion(
        [
            v1.PromptMessage(role="system", content=PROMPT_TEXT)
        ],
        model_name=MODEL_NAME,
        model_provider=MODEL_PROVIDER,
        template_format="NONE",
    ),
)

print(f"Prompt registered successfully!")
print(f"  Name      : {PROMPT_NAME}")
print(f"  Version ID: {prompt_version.id}")